# Tutorial: MNIST Baseline Walkthrough

Audience: first paper reproduction student.

Goal: see the baseline code step by step before treating it as an experiment.

This notebook is for learning. The reusable model and training script still live in `.py` files.

## Outline

1. Import the project code
2. Load one MNIST batch
3. Create the small CNN
4. Run one forward pass
5. Compute hard-label loss
6. Run one optimizer update
7. Then use the script for the real baseline run

## Step 1: Import project code

This cell makes sure the notebook can find the project's `src/` folder.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PROJECT_ROOT

WindowsPath('c:/Users/USER/Documents/雜貨專區/NUS UNI/NUS paper reproduction/knowledge-distillation-research')

## Step 2: Load one MNIST batch

A batch is a small group of images and labels.

In [2]:
from kd_research.data import create_mnist_loaders

train_loader, test_loader = create_mnist_loaders(
    data_dir=PROJECT_ROOT / "data",
    batch_size=8,
    download=False,
    num_workers=0,
)

images, labels = next(iter(train_loader))
images.shape, labels.shape

(torch.Size([8, 1, 28, 28]), torch.Size([8]))

Expected idea:

- `images.shape` should be `(8, 1, 28, 28)`
- `labels.shape` should be `(8,)`

## Step 3: Create the small CNN

This uses the model class from `src/kd_research/models/simple_cnn_student_model.py`.

In [3]:
from kd_research.models import SimpleCNNStudentModel

model = SimpleCNNStudentModel()
model

SimpleCNNStudentModel(
  (network): Sequential(
    (0): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Flatten(start_dim=1, end_dim=-1)
    (3): Linear(in_features=6272, out_features=10, bias=True)
  )
)

## Step 4: Run one forward pass

Forward pass means images go into the model and logits come out.

In [4]:
import torch

with torch.inference_mode():
    logits = model(images)

logits.shape

torch.Size([8, 10])

Expected idea:

- `logits.shape` should be `(8, 10)`
- 8 images
- 10 class scores for digits 0 to 9

## Step 5: Compute hard-label loss

Hard-label baseline uses the true labels from MNIST.

In [5]:
from torch import nn

loss_fn = nn.CrossEntropyLoss()
logits = model(images)
loss = loss_fn(logits, labels)

loss.item()

2.2656774520874023

The exact loss value is not important yet. At this point the model is not trained.

## Step 6: Run one optimizer update

This is one tiny training step, not a full experiment.

In [6]:
from torch.optim import Adam

optimizer = Adam(model.parameters(), lr=0.001)

optimizer.zero_grad()
logits = model(images)
loss = loss_fn(logits, labels)
loss.backward()
optimizer.step()

loss.item()

2.2656774520874023

## Step 7: Use the script for the real baseline run

After you understand the cells above, use the `.py` script for the full one-epoch run:

```cmd
.venv\Scripts\python.exe scripts\train_mnist_baseline.py --epochs 1 --batch-size 64 --learning-rate 0.001
```

## Exercise

In your own words, answer:

1. Why is `images.shape` `(8, 1, 28, 28)`?
2. Why is `logits.shape` `(8, 10)`?
3. Why is this still not Knowledge Distillation?

1. images.shape is (8, 1, 28, 28) because each batch has 8 pictures, each picture has 1 channel, and each picture is 28 height by 28 width.
2. logits.shape is (8, 10) because the batch has 8 pictures, and the model outputs 10 scores for each picture.
3. This is still not Knowledge Distillation because we are only using true labels first. We are not using a teacher model or soft targets yet.

Conclusion:
The hard-label baseline worked as a sanity check because the student CNN trained with true MNIST labels reached much higher accuracy(0.9589) than random guessing. This is not Knowledge Distillation yet because no teacher model or soft targets were used.

In [ ]:
import torch
import torch.nn.functional as F


def distillation_loss(student_logits, teacher_logits, temperature):
    student_log_probs = F.log_softmax(student_logits / temperature, dim=1)
    teacher_probs = F.softmax(teacher_logits / temperature, dim=1)


    return F.kl_div(student_log_probs, teacher_probs, reduction="batchmean",) * (temperature ** 2)



In [17]:
import torch
import torch.nn.functional as F


def distillation_loss(student_logits, teacher_logits, true_labels, temperature, alpha):
    student_log_probs = F.log_softmax(student_logits / temperature, dim=1)
    teacher_probs = F.softmax(teacher_logits / temperature, dim=1)


    soft_loss = F.kl_div(student_log_probs, teacher_probs, reduction="batchmean",) * (temperature ** 2)
    hard_loss = F.cross_entropy(student_logits, true_labels)
    total_loss = alpha * hard_loss + (1 - alpha) * soft_loss

    return hard_loss, soft_loss, total_loss


In [18]:
from pathlib import Path
import sys

import torch
import torch.nn.functional as F

project_root = Path(
    r"C:\Users\USER\Documents\雜貨專區\NUS UNI\NUS paper reproduction\knowledge-distillation-research"
)

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from kd_research.data import create_mnist_loaders
from kd_research.models import SimpleCNNTeacherModel

checkpoint_path = project_root / "checkpoints" / "mnist_teacher_baseline_001.pt"

print("notebook current folder:", Path.cwd())
print("checkpoint path:", checkpoint_path)
print("checkpoint exists:", checkpoint_path.exists())

temperature = 2.0
device = torch.device("cpu")

teacher = SimpleCNNTeacherModel().to(device)

checkpoint = torch.load(checkpoint_path, map_location=device)
teacher.load_state_dict(checkpoint["model_state_dict"])
teacher.eval()

_, test_loader = create_mnist_loaders(
    data_dir=project_root / "data",
    batch_size=4,
    download=False,
    num_workers=0,
)

images, labels = next(iter(test_loader))
images = images.to(device)

with torch.inference_mode():
    teacher_logits = teacher(images)
    teacher_soft_targets = F.softmax(teacher_logits / temperature, dim=1)

print("labels:", labels.tolist())
print("teacher_logits shape:", teacher_logits.shape)
print("teacher_soft_targets shape:", teacher_soft_targets.shape)
print("first soft target:")
print(teacher_soft_targets[0])
print("first soft target sum:", teacher_soft_targets[0].sum().item())

notebook current folder: c:\Users\USER\Documents\雜貨專區\NUS UNI\NUS paper reproduction\knowledge-distillation-research\notebooks
checkpoint path: C:\Users\USER\Documents\雜貨專區\NUS UNI\NUS paper reproduction\knowledge-distillation-research\checkpoints\mnist_teacher_baseline_001.pt
checkpoint exists: True
labels: [7, 2, 1, 0]
teacher_logits shape: torch.Size([4, 10])
teacher_soft_targets shape: torch.Size([4, 10])
first soft target:
tensor([8.5851e-05, 7.2415e-06, 2.8066e-04, 2.8677e-03, 9.6786e-06, 6.3678e-05,
        5.5013e-08, 9.9528e-01, 2.8107e-04, 1.1217e-03])
first soft target sum: 0.9999999403953552


hard_loss: 2.3176
soft_loss: 7.3515
total_loss: 3.8278
student updated: True


In [ ]:
from torch.optim import Adam
from kd_research.models import SimpleCNNStudentModel

alpha = 0.7 
temperature = 2.0

student = SimpleCNNStudentModel().to(device)
student.train()

train_loader, _ = create_mnist_loaders(data_dir=project_root / "data",
                                       batch_size=4, download=False, num_workers=0)

images, labels = next(iter(train_loader))
images = images.to(device)
labels = labels.to(device)

first_student_param = next(student.parameters())
#for comparison
before_update = first_student_param.detach().clone()

# use inference mode(not save gradient) save memory
with torch.inference_mode():
    teacher_logits = teacher(images)

student_logits = student(images)

hard_loss, soft_loss, total_loss = distillation_loss(
    student_logits=student_logits, teacher_logits=teacher_logits,
    true_labels=labels,
    temperature=temperature,
    alpha=alpha
)

optimizer = Adam(student.parameters(), lr=0.001)
optimizer.zero_grad
total_loss.backward()
optimizer.step()

after_update = first_student_param.detach().clone()
student_updated = not torch.equal(before_update, after_update)

print(f"hard_loss: {hard_loss.item():.4f}")
print(f"soft_loss: {soft_loss.item():.4f}")
print(f"total_loss: {total_loss.item():.4f}")
print("student updated:", student_updated)

hard_loss: 2.2883
soft_loss: 7.7387
total_loss: 3.9234
student updated: True
